# buffer-copy_-inplace — worked example 2: Reset a buffer to a broadcast scalar via copy_

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `buffer-copy_-inplace`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`tensor.copy_(src)` broadcasts `src` to the destination's shape, then writes into the destination's existing storage. A scalar (0-dim) or shape-(1,) source therefore fills every element of the buffer in place, preserving the buffer's identity. This is how you can 'reset' an EMA buffer to a constant without allocating a new tensor for the destination.

## Worked solution

**Goal.** Fill an existing `num_batches` buffer of shape `(5,)` with a single scalar value, in place, so its `data_ptr()` is unchanged.

**Step 1 — wrap the scalar as a tensor.** `src = t.tensor(value)` makes a 0-dim tensor. `copy_` accepts a 0-dim source and broadcasts it across the whole destination.

**Step 2 — copy in place.** `buf.copy_(src)` broadcasts the scalar to `buf.shape` and writes it into `buf`'s storage. Every element of `buf` becomes `value`, but `buf` is still the same object with the same memory — that is what makes it a valid in-place buffer reset.

**Why broadcasting matters.** Because `copy_` broadcasts, you don't have to pre-expand the scalar to shape `(5,)` yourself; the in-place op handles the right-aligned broadcast (0-dim is broadcastable to anything). The destination dtype is kept, so `value` is coerced to the buffer's dtype on the way in.

In [ ]:
def reset_buffer_to(buf: Tensor, value: float) -> None:
    src = t.tensor(value)
    buf.copy_(src)

t.manual_seed(0)
buf = t.arange(5, dtype=t.float32)
ptr_before = buf.data_ptr()
reset_buffer_to(buf, 3.5)
print("data_ptr preserved:", buf.data_ptr() == ptr_before)
print("buf:", buf)